# Part 1.3 The claims workflow in Studio

After this notebook you can run the finished claims workflow in LangSmith Studio and
read its path, its pause for human review, and its rejection of a bad photo.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

Studio shows the graph as a diagram and lights up each node as it runs.

<img src="../../assets/screenshots/p1-03-studio-graph.png" alt="Studio's graph view of claims right after the tear-01.jpg run: every node lit up, the conditional branches visible, and the input form in the left panel." width="900">

<img src="../../assets/screenshots/p1-03-studio-letter.png" alt="the same run's output panel showing status, payoutusd, and the reply letter." width="900">

Studio link: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024

## Build it

Nothing to code here. You start the server, open Studio, and file three claims the way a
passenger would: attach a photo and type one line.

### Start the server and open Studio

1. In a terminal at the repo root, run `mise run dev`. Wait until it prints that the
   server is ready on `http://127.0.0.1:2024`.
2. Open the Studio link above. In Chrome, allow **Local network access** when asked.
3. Pick the **`claims`** graph from the graph selector, then switch to the **Chat** view.

The photos are in `data/bag-photos/` (thumbnails at the end of this section).

### Run 1: a torn bag, paid automatically

Click **Upload files or images**, attach `tear-01.jpg`, and send:

> Passenger P-1001, flight KA 1182. My suitcase came back torn. It's worth $400.

The letter comes back as the reply. Switch to the **Graph** view to see the path it took:
`intake`, `validate_photo`, `assess_damage`, `check_history`, `apply_policy`,
`draft_letter`, `record_claim`.

<img src="../../assets/screenshots/p1-03-studio-run.png" alt="mid-run: the first nodes green, the rest still pending." width="900">

### Run 2: a crushed bag, paused for a person

In a new thread, attach `crushed-01.jpg` and send:

> Passenger P-1002, flight KA 1183. My hard-shell case was crushed. It was worth $800.

The payout is above $250, so the graph stops at `human_review`. Switch to the **Graph**
view to see the interrupt: the claim, the proposed payout, and the reasons. Approve it by
resuming with:

```json
{"approved": true, "payout_usd": 300, "note": "Approved at $300 after review."}
```

<img src="../../assets/screenshots/p1-03-studio-interrupt.png" alt="Studio paused on humanreview: the interrupt payload and the resume box." width="900">

### Run 3: not a bag, rejected

In a new thread, attach `car-01.jpg` and send:

> Passenger P-1003, flight KA 1184. Please cover this damage.

`validate_photo` says the photo is not luggage, so the graph takes the other branch to
`reject_letter` and never calls `assess_damage`. Send a message with no photo and
`intake` rejects it the same way.

Prefer typing fields? The **Graph** view's input form takes them directly, for example
`{"photo": "tear-01.jpg", "passenger_id": "P-1001", "flight_number": "KA 1182",
"declared_value_usd": 400}`. The photo can be a file name from `data/bag-photos`.

<img src="../../assets/screenshots/p1-03-studio-reject.png" alt="the car-01.jpg run: rejectletter lit, assessdamage untouched, and the rejection letter in the output." width="900">

Use a new `passenger_id` for each run you repeat. The workflow remembers a passenger's
earlier claims, and a third claim goes to review (Part 1.9).

The photos you can use:

In [ ]:
from IPython.display import HTML, display

from kea.claims.images import PHOTO_DIR, image_to_data_url

print("Studio:", "https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024")
tiles = "".join(
    f'<figure style="margin:4px;text-align:center"><img src="{image_to_data_url(p)}" '
    f'width="150"><figcaption>{p.name}</figcaption></figure>'
    for p in sorted(PHOTO_DIR.glob("*.jpg"))
)
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{tiles}</div>'))

## Under the hood

- `mise run dev` runs `langgraph dev`, a local **Agent Server** that serves the graphs
  named in `langgraph.json` (`claims` and `rebooking`). Studio is a web app that talks to
  that server on port 2024.
- The server supplies the **checkpointer** and **store**. That is why the pause in run 2
  can wait for you, and why run history persists, with no code in the graph for it.
- Studio's Chat view sends the photo inside a chat message. The first node, `intake`,
  takes the image out of the message (`photo_from_content` in `kea/claims/images.py`) and
  asks the model for the passenger, flight, and bag value in the text. A claim sent as
  fields, as the notebooks and the experiments in Part 1.12 do, skips that step.
- The reply letter is also added to `messages`, which is what the Chat view displays.
- Each Studio run is also a trace in LangSmith, so you can open it with the same tools
  as the next notebook.

Docs: [LangSmith Studio](https://docs.langchain.com/langsmith/studio).

## Recap

- `mise run dev` starts the server; Studio runs the graph against it.
- A normal claim runs straight through; a risky one pauses for a person.
- A conditional edge sends a non-luggage photo to a rejection.
- You've now seen the whole workflow. Next, you build its parts.

**Next:** [Part 1.4 Nodes and edges](04_nodes_and_edges.ipynb)